In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()  # loads ANTHROPIC_API_KEY from .env automatically

client = Anthropic()

In [4]:
from anthropic import Anthropic
from anthropic.types import MessageParam

def llm(prompt):
    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=1024,
        messages=[
            MessageParam(role="user", content=prompt)
        ]
    )
    return response.content[0].text

In [14]:
def llm(prompt):
    response = client.messages.create(
        model="claude-sonnet-4-5",  # correct model name
        max_tokens=1024,
        messages=[{"role": "user", "content": prompt}] # type: ignore
    )
    return response.content[0].text

In [15]:
llm("Hello World")

'Hello! 👋 Welcome! How can I help you today?'

In [19]:
question = 'I just discovered the course. Can I join now?'
answer = llm(question)
print(answer)

context = '''
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
'''

prompt = f'''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
'''

answer = llm(prompt)
print(answer)

I'd be happy to help you! However, I need a bit more information to give you the best answer:

1. **Which course** are you interested in joining?
2. **When does it start** (or when did it start)?
3. **What platform or institution** is offering it?

In general, here are some common scenarios:

- **Online courses** (Coursera, edX, Udemy, etc.) - Many allow you to join anytime or have rolling enrollment
- **University courses** - These usually have add/drop deadlines (often 1-2 weeks into the semester)
- **Bootcamps or cohort-based courses** - May have strict start dates but sometimes accept late joiners

**What you can do right now:**
- Check the course website for enrollment deadlines
- Contact the instructor or course administrator directly
- Look for a "late enrollment" or "join now" option

If you share more details about the specific course, I can give you more targeted advice!
Based on the context provided, **yes, you can still join the course now**. 

However, if you want to recei

In [22]:
import requests

docs_url = 'https://datatalks.club/faq/json/courses.json'
# make a get request to url
response = requests.get(docs_url)

# parse the response body as JSON. give sus a python list/dict
courses_raw = response.json()

print(courses_raw)


documents = []
url_prefix = 'https://datatalks.club/faq' # base url.

for course in courses_raw:
    # build the full URL for this course's FAQ
    course_url = f'{url_prefix}{course['path']}'

# fetch the FAQ data for this specific course.
    course_response = requests.get(course_url)

    # if request failed (400, 500 etc) stop and raise an error immediately.
    course_response.raise_for_status()

    #parse this course FAQ response as JSON.
    course_data = course_response.json()

# Add all FAQ documents from this course into our master list.
    # extend() adds each item individually
    documents.extend(course_data)

# shows the number of FAQ documents which we collect across all courses.
len(documents)

[{'course': 'machine-learning-zoomcamp', 'course_name': 'ML Zoomcamp', 'path': '/json/machine-learning-zoomcamp.json', 'questions_count': 472}, {'course': 'llm-zoomcamp', 'course_name': 'LLM Zoomcamp', 'path': '/json/llm-zoomcamp.json', 'questions_count': 79}, {'course': 'data-engineering-zoomcamp', 'course_name': 'Data Engineering Zoomcamp', 'path': '/json/data-engineering-zoomcamp.json', 'questions_count': 402}, {'course': 'mlops-zoomcamp', 'course_name': 'MLOps Zoomcamp', 'path': '/json/mlops-zoomcamp.json', 'questions_count': 255}]


1208

In [23]:
documents[0]

{'id': '0e38656cfb',
 'course': 'machine-learning-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'How do I submit homework?',
 'answer': "- Do the tasks locally\n- Publish your code (e.g., in your own GitHub repo)\n- Submit your answers via the homework form and include the URL to your code\n- You will see the answers only after the deadline\n- Homeworks are in the cohorts folder, e.g. for 2025 it's [`cohorts/2025`](https://github.com/DataTalksClub/machine-learning-zoomcamp/tree/master/cohorts/2025)\n- The forms for submitting the homework are in the [course management platform](https://courses.datatalks.club/)"}

In [24]:
documents[1100]

{'id': '841966c903',
 'course': 'mlops-zoomcamp',
 'section': 'Module 3: Orchestration',
 'question': 'Where is the FAQ for Prefect questions?',
 'answer': '[Here](https://docs.google.com/document/d/1Nyktf7WoRec5lDUBREXL5zLI1Edbw9_R8e45fDn4KB8/edit?usp=sharing)'}

In [27]:
from minsearch import Index

index = Index(
    text_fields=['question', 'section', 'answer'],
    keyword_fields=['course']
)

index.fit(documents)